# Train and Inference Notebook

This notebook combines the functionality of train.py and inference.py into a single interactive workflow.

## Import Required Libraries

Import all necessary libraries such as TensorFlow, PyTorch, NumPy, pandas, and others used in both train.py and inference.py.

In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import pickle

# Deep Learning Libraries
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, models, optimizers
    print(f"TensorFlow version: {tf.__version__}")
except ImportError:
    print("TensorFlow not installed")

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, Dataset
    print(f"PyTorch version: {torch.__version__}")
except ImportError:
    print("PyTorch not installed")

# Scikit-learn for additional utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set random seeds for reproducibility
np.random.seed(42)
try:
    tf.random.set_seed(42)
except:
    pass
try:
    torch.manual_seed(42)
except:
    pass

print("All libraries imported successfully!")

## Define Helper Functions

Combine any utility functions from train.py and inference.py into a single section.

In [ ]:
# Helper Functions

def create_directory(path):
    """Create directory if it doesn't exist"""
    Path(path).mkdir(parents=True, exist_ok=True)
    print(f"Directory ensured: {path}")

def save_metrics(metrics, filepath):
    """Save training metrics to JSON file"""
    with open(filepath, 'w') as f:
        json.dump(metrics, f, indent=4)
    print(f"Metrics saved to: {filepath}")

def load_metrics(filepath):
    """Load metrics from JSON file"""
    with open(filepath, 'r') as f:
        metrics = json.load(f)
    print(f"Metrics loaded from: {filepath}")
    return metrics

def plot_training_history(history, save_path=None):
    """Plot training history"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot loss
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'], label='Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Plot accuracy
    axes[1].plot(history['train_acc'], label='Train Accuracy')
    axes[1].plot(history['val_acc'], label='Validation Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
        print(f"Training history plot saved to: {save_path}")
    
    plt.show()

def normalize_data(X_train, X_test):
    """Normalize data using StandardScaler"""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

print("Helper functions defined successfully!")

## Load and Preprocess Data

Include data loading and preprocessing steps from train.py.

In [ ]:
# Load and Preprocess Data

# Configuration
DATA_PATH = "f:/BUET_TEST/data"
MODEL_PATH = "f:/BUET_TEST/models"
OUTPUT_PATH = "f:/BUET_TEST/output"

# Create necessary directories
create_directory(DATA_PATH)
create_directory(MODEL_PATH)
create_directory(OUTPUT_PATH)

# Example: Load your dataset here
# For demonstration, we'll create synthetic data
print("Loading data...")

# Generate synthetic classification data
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=3,
    random_state=42
)

print(f"Data shape: X={X.shape}, y={y.shape}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Test set: X_test={X_test.shape}, y_test={y_test.shape}")

# Normalize the data
X_train_scaled, X_test_scaled, scaler = normalize_data(X_train, X_test)

# Save the scaler for inference
scaler_path = f"{MODEL_PATH}/scaler.pkl"
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"Scaler saved to: {scaler_path}")

# Convert labels to categorical (for neural networks)
n_classes = len(np.unique(y))
y_train_cat = keras.utils.to_categorical(y_train, n_classes)
y_test_cat = keras.utils.to_categorical(y_test, n_classes)

print("Data loaded and preprocessed successfully!")

## Define Model Architecture

Define the model architecture as implemented in train.py.

In [ ]:
# Define Model Architecture

def create_model(input_dim, n_classes, hidden_units=[64, 32]):
    """
    Create a neural network model
    
    Args:
        input_dim: Number of input features
        n_classes: Number of output classes
        hidden_units: List of hidden layer units
    
    Returns:
        Compiled Keras model
    """
    model = models.Sequential()
    
    # Input layer
    model.add(layers.Input(shape=(input_dim,)))
    
    # Hidden layers
    for units in hidden_units:
        model.add(layers.Dense(units, activation='relu'))
        model.add(layers.Dropout(0.3))
    
    # Output layer
    model.add(layers.Dense(n_classes, activation='softmax'))
    
    # Compile model
    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Create the model
input_dim = X_train_scaled.shape[1]
model = create_model(input_dim, n_classes, hidden_units=[64, 32, 16])

# Display model summary
print("Model Architecture:")
model.summary()

## Train the Model

Add the training loop and logic from train.py, including loss calculation and optimization.

In [ ]:
# Train the Model

# Training configuration
EPOCHS = 50
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2

print(f"Starting training for {EPOCHS} epochs...")

# Define callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Train the model
history = model.fit(
    X_train_scaled, y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

print("Training completed!")

# Extract training history
training_history = {
    'train_loss': history.history['loss'],
    'val_loss': history.history['val_loss'],
    'train_acc': history.history['accuracy'],
    'val_acc': history.history['val_accuracy']
}

# Plot training history
plot_training_history(training_history, save_path=f"{OUTPUT_PATH}/training_history.png")

## Save the Trained Model

Include the code to save the trained model to disk as implemented in train.py.

In [ ]:
# Save the Trained Model

# Save model in different formats
model_save_path = f"{MODEL_PATH}/trained_model.h5"
model.save(model_save_path)
print(f"Model saved to: {model_save_path}")

# Save model in SavedModel format (recommended for TensorFlow)
model_saved_format_path = f"{MODEL_PATH}/saved_model"
model.save(model_saved_format_path)
print(f"Model saved in SavedModel format to: {model_saved_format_path}")

# Save training history
history_path = f"{MODEL_PATH}/training_history.json"
save_metrics(training_history, history_path)

# Save model configuration
model_config = {
    'input_dim': input_dim,
    'n_classes': n_classes,
    'hidden_units': [64, 32, 16],
    'epochs_trained': len(training_history['train_loss']),
    'final_train_loss': training_history['train_loss'][-1],
    'final_val_loss': training_history['val_loss'][-1],
    'final_train_acc': training_history['train_acc'][-1],
    'final_val_acc': training_history['val_acc'][-1]
}

config_path = f"{MODEL_PATH}/model_config.json"
save_metrics(model_config, config_path)

print("All model artifacts saved successfully!")

## Load the Trained Model

Add the code to load the trained model from disk as implemented in inference.py.

In [ ]:
# Load the Trained Model

print("Loading trained model for inference...")

# Load the model
loaded_model = keras.models.load_model(model_save_path)
print(f"Model loaded from: {model_save_path}")

# Load the scaler
with open(scaler_path, 'rb') as f:
    loaded_scaler = pickle.load(f)
print(f"Scaler loaded from: {scaler_path}")

# Load model configuration
loaded_config = load_metrics(config_path)
print(f"Model configuration: {loaded_config}")

# Verify model architecture
print("\nLoaded Model Architecture:")
loaded_model.summary()

## Perform Inference

Include the inference logic from inference.py to make predictions using the trained model.

In [ ]:
# Perform Inference

print("Performing inference on test data...")

# Make predictions on test data
y_pred_proba = loaded_model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_proba, axis=1)

print(f"Predictions shape: {y_pred.shape}")
print(f"First 10 predictions: {y_pred[:10]}")
print(f"First 10 actual labels: {y_test[:10]}")

# Example: Single sample prediction
print("\n--- Single Sample Inference ---")
sample_idx = 0
single_sample = X_test_scaled[sample_idx:sample_idx+1]
single_pred_proba = loaded_model.predict(single_sample, verbose=0)
single_pred = np.argmax(single_pred_proba, axis=1)[0]

print(f"Sample index: {sample_idx}")
print(f"Predicted class: {single_pred}")
print(f"Prediction probabilities: {single_pred_proba[0]}")
print(f"Actual class: {y_test[sample_idx]}")

# Batch inference example
print("\n--- Batch Inference ---")
batch_size = 10
batch_samples = X_test_scaled[:batch_size]
batch_pred_proba = loaded_model.predict(batch_samples, verbose=0)
batch_pred = np.argmax(batch_pred_proba, axis=1)

print(f"Batch predictions: {batch_pred}")
print(f"Batch actual labels: {y_test[:batch_size]}")

print("\nInference completed successfully!")

## Evaluate Inference Results

Add code to evaluate and visualize the inference results, if applicable, from inference.py.

In [ ]:
# Evaluate Inference Results

print("Evaluating model performance on test data...")

# Calculate accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {test_accuracy:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f'Class {i}' for i in range(n_classes)],
            yticklabels=[f'Class {i}' for i in range(n_classes)])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/confusion_matrix.png")
plt.show()

# Prediction probability distribution
plt.figure(figsize=(10, 6))
for i in range(n_classes):
    plt.hist(y_pred_proba[:, i], bins=30, alpha=0.5, label=f'Class {i}')
plt.xlabel('Prediction Probability')
plt.ylabel('Frequency')
plt.title('Distribution of Prediction Probabilities')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/probability_distribution.png")
plt.show()

# Save evaluation results
evaluation_results = {
    'test_accuracy': float(test_accuracy),
    'confusion_matrix': cm.tolist(),
    'classification_report': classification_report(y_test, y_pred, output_dict=True)
}

eval_path = f"{OUTPUT_PATH}/evaluation_results.json"
save_metrics(evaluation_results, eval_path)

print("\nEvaluation completed and results saved!")

## Summary

This notebook successfully combines the training and inference workflows:

1. **Data Preparation**: Loaded and preprocessed data with normalization
2. **Model Training**: Built and trained a neural network with early stopping and learning rate reduction
3. **Model Saving**: Saved the trained model, scaler, and configuration
4. **Model Loading**: Loaded the saved model and preprocessing artifacts
5. **Inference**: Performed predictions on test data (single sample and batch)
6. **Evaluation**: Evaluated model performance with metrics and visualizations

All outputs are saved to `f:/BUET_TEST/output/` and model artifacts to `f:/BUET_TEST/models/`.